# Robustness on gLV heteroclinic channels

Scaling with the number of species $N$ (Appendix I.3, Fig. 5) and sensitivity to the finite-difference lag $\tau$ (Appendix K, Fig. 8).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import normalized_mutual_info_score

RESULTS_DIR = "results"  # output folder for figures
DATA_DIR = "data"  # output folder for intermediate results
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
plt.style.use("tableau-colorblind10")
plt.rcParams.update(
    {
        "figure.autolayout": True,
        "font.size": 10,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "font.family": "serif",
        "font.serif": "cmr10",
        "mathtext.fontset": "cm",
        "axes.formatter.use_mathtext": True,
    }
)


# Past/future increments with lag tau
def finite_difference_vectors(x, tau=1):
    fd = x[:, tau:] - x[:, :-tau]
    return fd[:, :-1], fd[:, 1:]

In [ ]:
# Offline LPD (Algorithm 2), GPU implementation
def lpd_offline_torch(
    X_np,
    Y_np,
    k=20,
    Nz_max=100,
    a_z=0.1,
    a_W=0.2,
    a_M=0.2,
    eps=1e-5,
    Nmax=3000,
    device=None,
):

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    X = torch.tensor(X_np, dtype=torch.float32, device=device)
    Y = torch.tensor(Y_np, dtype=torch.float32, device=device)

    Nx, T = X.shape
    Ny = Y.shape[0]

    X_t = X.T
    Y_t = Y.T

    Cxx = (X @ X.T) / T
    Cyy = (Y @ Y.T) / T
    Cxy = (X @ Y.T) / T

    I_x = torch.eye(Nx, device=device)
    I_y = torch.eye(Ny, device=device)

    Cxx_inv = torch.linalg.inv(Cxx + eps * I_x)
    Cyy_inv = torch.linalg.inv(Cyy + eps * I_y)

    m = k

    def orthogonal_matrix(dim):
        Q, _ = torch.linalg.qr(torch.randn(dim, dim, device=device))
        return Q

    M = torch.eye(m, device=device)

    W = (
        orthogonal_matrix(m) @ torch.eye(m, Nx, device=device) @ orthogonal_matrix(Nx)
    ) * 0.1

    Z = torch.rand(T, m, device=device)
    Z = torch.relu(Z)

    Z = Z / torch.clamp(torch.norm(Z, dim=0, keepdim=True), min=1e-10)

    err_list = []
    obs_list = []

    rf = torch.randn_like(W)

    with torch.no_grad():
        for it in range(Nmax):

            for _ in range(Nz_max):
                Z_new = (W @ X_t.T - M @ Z.T).T
                Z = a_z * Z_new + (1 - a_z) * Z
                Z = torch.relu(Z)

            Z = Z / torch.clamp(torch.norm(Z, dim=0, keepdim=True), min=1e-10)

            M_new = Z.T @ Z
            M = a_M * M_new + (1 - a_M) * M

            Czx = (Z.T @ X_t) / torch.sqrt(
                torch.tensor(T, device=device, dtype=torch.float32)
            )

            W_new = Czx @ Cxx_inv @ Cxy @ Cyy_inv @ Cxy.T @ Cxx_inv

            W = a_W * W_new + (1 - a_W) * W

            MW = torch.linalg.solve(M + eps * torch.eye(m, device=device), W)

            error = torch.sum(torch.abs(rf - MW)) / (torch.sum(torch.abs(rf)) + 1e-12)
            rf = MW.clone()

            err_list.append(error.item())
            obs_list.append((torch.norm(M).item(), torch.norm(W).item()))

            if it % 300 == 0:
                print(f"[Iter {it}] error = {error.item():.3e}")

    Z_out = Z.T.detach().cpu().numpy()
    W_out = W.detach().cpu().numpy()
    M_out = M.detach().cpu().numpy()

    return Z_out, (W_out, M_out), (err_list, obs_list)

## Scaling with system size $N$ (Fig. 5)

In [ ]:
# Cyclic heteroclinic channel between single-species saddles
def build_shc_matrix(N, inhib=1.5, self_inhib=1.0, forward=0.7):
    """Simple cyclic SHC"""
    A = np.ones((N, N)) * inhib
    np.fill_diagonal(A, self_inhib)

    for i in range(N):
        A[i, (i + 1) % N] = forward

    return A


def transition_from_A(A):
    N = A.shape[0]
    P = np.zeros((N, N))
    for i in range(N):
        row = A[i].copy()
        row[i] = np.inf
        j = np.argmin(row)
        P[i, j] = 1
    return P


# Match learned filters to ground-truth transitions (Hungarian algorithm)
def align_MW(MW, P_ref):
    MW_norm = MW / (np.linalg.norm(MW, axis=1, keepdims=True) + 1e-12)
    similarity = MW_norm @ P_ref.T
    row_ind, col_ind = linear_sum_assignment(-similarity)
    MW_aligned = MW_norm[row_ind]
    MW_aligned = MW_aligned[np.argsort(col_ind)]
    return MW_aligned


def transition_accuracy(P1, P2):
    pred = np.argmax(P1, axis=1)
    true = np.argmax(P2, axis=1)
    return np.mean(pred == true)

In [ ]:
N_list = [3, 5, 10, 25, 50, 100, 200]
sigma_list = [0.01, 0.05, 0.1]
n_realizations = 50

save_path = os.path.join(DATA_DIR, "benchmark_results.pkl")
csv_path = os.path.join(DATA_DIR, "benchmark_results.csv")

results = []


def transition_nmi(P_pred, P_true):
    pred = np.argmax(P_pred, axis=1)
    true = np.argmax(P_true, axis=1)
    return normalized_mutual_info_score(true, pred)


def run_experiment(N, sigma, seed=None):

    if seed is not None:
        np.random.seed(seed)

    T = 5000
    dt = 0.1

    A = build_shc_matrix(N)

    x0 = np.zeros(N)
    x0[0] = 1.0
    x0 += 0.01

    num_steps = int(T / dt)
    X = np.zeros((N, num_steps))
    X[:, 0] = x0

    for k in range(1, num_steps):
        x = X[:, k - 1]

        dx = x * (1 - A @ x)
        dx += sigma * np.random.randn(N)

        X[:, k] = np.clip(x + dx * dt, 0, None)

    trajs = X

    N_stat = 1_000
    X_fd, Y_fd = finite_difference_vectors(trajs[:, N_stat:])
    N_data = 19_000
    X_in = X_fd[:, :N_data]
    Y_in = Y_fd[:, :N_data]

    k = N

    Z, (W_lpd, M_lpd), _ = lpd_offline_torch(
        X_in, Y_in, k=k, Nz_max=60, a_z=0.1, a_W=0.2, a_M=0.2, eps=1e-4, Nmax=1000
    )

    MW = np.linalg.solve(M_lpd + 1e-5 * np.eye(M_lpd.shape[0]), W_lpd)

    MW = MW / (np.linalg.norm(MW, axis=1, keepdims=True) + 1e-12)

    P_theory = transition_from_A(A)

    MW_aligned = align_MW(MW, P_theory)

    acc = transition_accuracy(MW_aligned, P_theory)
    nmi = transition_nmi(MW_aligned, P_theory)

    return acc, nmi

run_id = 0

for N in N_list:
    for sigma in sigma_list:

        for r in range(n_realizations):

            run_id += 1
            print(f"Run {run_id}: N={N}, sigma={sigma}, r={r}")

            try:
                acc, nmi = run_experiment(N, sigma, seed=run_id)

                results.append(
                    {
                        "N": N,
                        "sigma": sigma,
                        "realization": r,
                        "accuracy": acc,
                        "nmi": nmi,
                    }
                )

                df = pd.DataFrame(results)
                df.to_pickle(save_path)
                df.to_csv(csv_path, index=False)

            except Exception as e:
                print(f"Failed run N={N}, sigma={sigma}, r={r}: {e}")
                continue

df = pd.DataFrame(results)
print(df.head())

df.to_pickle(save_path)
df.to_csv(csv_path, index=False)

print("Saved results to:", save_path, "and", csv_path)

In [ ]:
df = pd.read_pickle(os.path.join(DATA_DIR, "benchmark_results.pkl"))
df["sigma"] = df["sigma"].astype(float)

df_grouped = df.groupby(["N", "sigma"])["nmi"].agg(mean="mean", std="std").reset_index()

colors = ["royalblue", "firebrick", "seagreen"]
markers = ["o", "s", "D"]

sigma_list = sorted(df_grouped["sigma"].unique())
legend_labels = [rf"$\sigma = {s}$" for s in sigma_list]

fig, ax = plt.subplots(figsize=(8 / 2.54, 6 / 2.54))

lines = []

for idx, sigma in enumerate(sigma_list):
    df_subset = df_grouped[df_grouped["sigma"] == sigma].sort_values("N")

    l = ax.errorbar(
        df_subset["N"],
        df_subset["mean"],
        yerr=df_subset["std"],
        marker=markers[idx],
        color=colors[idx],
        label=legend_labels[idx],
        capsize=2,
        elinewidth=1.0,
        markersize=4,
    )
    lines.append(l)

ax.set_xlabel("N")
ax.set_ylabel("NMI")

ax.set_xscale("log")

ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.6, 1.05),
    ncol=len(legend_labels),
    frameon=False,
)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "nmi_vs_N.png"), dpi=600, bbox_inches="tight")
plt.show()

## Effect of the finite-difference lag $\tau$ (Fig. 8)

In [ ]:
N = 10

tau_list = [1, 5, 10, 15, 20, 25]
sigma_list = [0.01, 0.05, 0.1]

n_realizations = 20

save_path = os.path.join(DATA_DIR, "benchmark_tau_results.pkl")
csv_path = os.path.join(DATA_DIR, "benchmark_tau_results.csv")

results = []


def run_experiment(tau, sigma, seed=None):

    if seed is not None:
        np.random.seed(seed)

    T = 2000
    dt = 0.1

    A = build_shc_matrix(N)

    x0 = np.zeros(N)
    x0[0] = 1.0
    x0 += 0.01

    num_steps = int(T / dt)
    X = np.zeros((N, num_steps))
    X[:, 0] = x0

    for k in range(1, num_steps):
        x = X[:, k - 1]

        dx = x * (1 - A @ x)
        dx += sigma * np.random.randn(N)

        X[:, k] = np.clip(x + dx * dt, 0, None)

    trajs = X

    X_fd, Y_fd = finite_difference_vectors(trajs[:, 1000:], tau=tau)

    X_in = X_fd
    Y_in = Y_fd

    k = N

    Z, (W_lpd, M_lpd), _ = lpd_offline_torch(
        X_in, Y_in, k=k, Nz_max=100, a_z=0.1, a_W=0.2, a_M=0.2, eps=1e-4, Nmax=3000
    )

    MW = np.linalg.solve(M_lpd + 1e-5 * np.eye(M_lpd.shape[0]), W_lpd)

    MW = MW / (np.linalg.norm(MW, axis=1, keepdims=True) + 1e-12)

    P_theory = transition_from_A(A)

    MW_aligned = align_MW(MW, P_theory)

    acc = transition_accuracy(MW_aligned, P_theory)
    nmi = transition_nmi(MW_aligned, P_theory)

    return acc, nmi

run_id = 0

for tau in tau_list:
    for sigma in sigma_list:

        for r in range(n_realizations):
            r = r + 1
            run_id += 1
            print(f"Run {run_id}: tau={tau}, sigma={sigma}, r={r}")

            try:
                acc, nmi = run_experiment(tau, sigma, seed=run_id)

                results.append(
                    {
                        "tau": tau,
                        "sigma": sigma,
                        "realization": r,
                        "accuracy": acc,
                        "nmi": nmi,
                    }
                )

                df = pd.DataFrame(results)
                df.to_pickle(save_path)
                df.to_csv(csv_path, index=False)

            except Exception as e:
                print(f"Failed tau={tau}, sigma={sigma}, r={r}: {e}")
                continue

df = pd.DataFrame(results)
print(df.head())

df.to_pickle(save_path)
df.to_csv(csv_path, index=False)

print("Saved to:", save_path)

In [ ]:
sigma_colors = {0.01: "royalblue", 0.05: "firebrick", 0.1: "seagreen"}
markers = ["s", "s", "s"]

fig, ax = plt.subplots(figsize=(3.0, 1.6))

df = pd.read_pickle(os.path.join(DATA_DIR, "benchmark_tau_results.pkl"))
df["sigma"] = df["sigma"].astype(float)

df_tau = df.groupby(["tau", "sigma"])["nmi"].agg(mean="mean", std="std").reset_index()

for i, sigma in enumerate(sorted(sigma_colors)):
    df_s = df_tau[df_tau["sigma"] == sigma]
    ax.errorbar(
        df_s["tau"],
        df_s["mean"],
        yerr=df_s["std"],
        marker=markers[i],
        color=sigma_colors[sigma],
        capsize=2,
    )

ax.set_xlabel(r"Finite differences lag $\tau$")
ax.set_ylabel("NMI")
ax.set_ylim(0.6, 1.05)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "nmi_vs_tau.png"), dpi=600, bbox_inches="tight")
plt.show()